# Market Candlestick Viewer

Quick viewer for OHLCV parquet data under `data/market/<SYMBOL>/history_*.parquet`.
- Supports multiple symbols
- Shows candlesticks
- Shows volume (toggle)

In [ ]:
from pathlib import Path
from datetime import date, timedelta
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, Markdown

DATA_ROOT = Path('data/market')

def _normalize_symbol(symbol: str) -> str:
    return symbol.strip().upper()

def load_symbol_history(symbol: str, start=None, end=None, data_root: Path = DATA_ROOT) -> pd.DataFrame:
    symbol = _normalize_symbol(symbol)
    symbol_dir = data_root / symbol
    if not symbol_dir.exists():
        raise FileNotFoundError(f'No directory for symbol: {symbol} ({symbol_dir})')

    files = sorted(symbol_dir.glob('history_*.parquet'))
    if not files:
        raise FileNotFoundError(f'No history parquet files found for {symbol}')

    frames = [pd.read_parquet(f) for f in files]
    df = pd.concat(frames, ignore_index=True)

    if 'Date' not in df.columns:
        raise ValueError(f'Unexpected schema for {symbol}: missing Date column')

    df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
    df = df.dropna(subset=['Date']).drop_duplicates(subset=['Date']).sort_values('Date')

    if start is not None:
        df = df[df['Date'] >= pd.to_datetime(start)]
    if end is not None:
        df = df[df['Date'] <= pd.to_datetime(end)]

    if df.empty:
        raise ValueError(f'No data in selected date window for {symbol}')

    return df.reset_index(drop=True)

def plot_candles(symbol: str, df: pd.DataFrame, show_volume: bool = True):
    has_volume = show_volume and ('Volume' in df.columns)

    if has_volume:
        fig = make_subplots(
            rows=2, cols=1, shared_xaxes=True,
            row_heights=[0.72, 0.28], vertical_spacing=0.04,
            subplot_titles=[f'{symbol} Candlestick', 'Volume']
        )
    else:
        fig = make_subplots(rows=1, cols=1, subplot_titles=[f'{symbol} Candlestick'])

    fig.add_trace(
        go.Candlestick(
            x=df['Date'],
            open=df['Open'],
            high=df['High'],
            low=df['Low'],
            close=df['Close'],
            name=symbol
        ),
        row=1, col=1
    )

    if has_volume:
        up = df['Close'] >= df['Open']
        colors = ['#1f7a1f' if is_up else '#b22222' for is_up in up]
        fig.add_trace(
            go.Bar(
                x=df['Date'],
                y=df['Volume'],
                marker_color=colors,
                name='Volume'
            ),
            row=2, col=1
        )

    fig.update_layout(
        template='plotly_white',
        height=700 if has_volume else 520,
        xaxis_rangeslider_visible=False,
        title=f'{symbol} OHLCV'
    )
    fig.update_yaxes(title_text='Price', row=1, col=1)
    if has_volume:
        fig.update_yaxes(title_text='Volume', row=2, col=1)

    fig.show()

In [ ]:
# Interactive controls
import ipywidgets as widgets
from IPython.display import clear_output

symbols_input = widgets.Text(
    value='BTC-USD, ETH-USD, GC=F',
    description='Symbols',
    layout=widgets.Layout(width='700px')
)
start_input = widgets.DatePicker(description='Start', value=date.today() - timedelta(days=365))
end_input = widgets.DatePicker(description='End', value=date.today())
volume_input = widgets.Checkbox(value=True, description='Show Volume')
run_button = widgets.Button(description='Plot', button_style='primary')
out = widgets.Output()

def _parse_symbols(raw: str):
    return [s.strip().upper() for s in raw.split(',') if s.strip()]

def on_run(_):
    symbols = _parse_symbols(symbols_input.value)
    with out:
        clear_output(wait=True)
        if not symbols:
            display(Markdown('**No symbols provided.**'))
            return

        display(Markdown(f'### Rendering {len(symbols)} symbol(s)'))
        for sym in symbols:
            try:
                df = load_symbol_history(sym, start=start_input.value, end=end_input.value)
                display(Markdown(f'#### {sym}  ({df["Date"].min().date()} to {df["Date"].max().date()}, rows={len(df)})'))
                plot_candles(sym, df, show_volume=volume_input.value)
            except Exception as e:
                display(Markdown(f'- `{sym}`: {e}'))

run_button.on_click(on_run)
display(widgets.VBox([
    symbols_input,
    widgets.HBox([start_input, end_input, volume_input, run_button]),
    out
]))

In [ ]:
# Optional quick run without widgets
# symbols = ['BTC-USD', 'ETH-USD', 'GC=F']
# for s in symbols:
#     df = load_symbol_history(s, start='2025-01-01', end='2026-02-28')
#     plot_candles(s, df, show_volume=True)